# Hypothesis-driven investment research backtest

A one-shot LangGraph interprets the hypothesis at each eligible monthly rebalance. Views become deterministic long-only research-strength tilts, **not Black-Litterman weights**. Early rebalances use explicitly recorded equal-weight warm-up allocations until 64 observations are available.

Select the repository Python environment and install the backend requirements. Set `OPENAI_API_KEY` in your environment or a local `.env` (never in this notebook). The default run covers a short period to limit paid calls; change the dates and hypothesis below. EPS and rates are synthetic. This is an integration demo, not evidence of historically valid investment performance.

In [1]:
import sys
from pathlib import Path


def _find_backend_dir(start: Path) -> Path:
    for p in [start, *start.parents]:
        for candidate in (p, p / 'backend'):
            if (candidate / 'app' / 'services' / 'bl_backtest').is_dir():
                return candidate
    raise RuntimeError('Could not locate backend directory from ' + str(start))


_BACKEND_DIR = _find_backend_dir(Path.cwd())
if str(_BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(_BACKEND_DIR))

import bt
import pandas as pd
import plotly.graph_objects as go
from dotenv import load_dotenv

from app.services.bl_backtest.run_example import (
    HYPOTHESIS, _NON_EQUITY_ASSETS,
    build_agentic_strategy, build_equal_weight_strategy,
)
from app.services.bl_backtest.data_interface.load import load_research_data
from app.services.bl_backtest.research_weights import make_research_weight_fn

load_dotenv(_BACKEND_DIR / '.env')
load_dotenv(_BACKEND_DIR.parent / '.env')

# Edit these settings before executing the next cell.
RESEARCH_HYPOTHESIS = HYPOTHESIS
START_DATE = '2021-03-19'
END_DATE = '2021-09-30'
LLM_CLIENT = None  # None uses OpenAI; tests can inject a fake client.
USAGE_DB_PATH = None  # None uses the existing LLM usage database.

COLOUR_AGENTIC = '#60a5fa'
COLOUR_EQUAL = '#f59e0b'


## Run monthly hypothesis research

All prices come from the existing CSV evidence exports. Each callback receives the actual rebalance timestamp, selects only evidence through that date, and invokes the graph once after warm-up. The tilt score is `1 + 0.5 * direction_sign * magnitude * confidence`, normalized across the selected securities. Provider/validation failures stop the run rather than silently returning mock weights.

In [ ]:
from app.services.bl_backtest.schema import ResearchRequest
from app.services.bl_backtest.data_interface.process import prepare_context

data = load_research_data()
universe = [a for a in data.metadata['all_assets'] if a not in _NON_EQUITY_ASSETS]
prepare_context(data, ResearchRequest(hypothesis=RESEARCH_HYPOTHESIS, universe=universe))
start, end = pd.Timestamp(START_DATE), pd.Timestamp(END_DATE)
if not data.prices.index[0] <= start <= end <= data.prices.index[-1]:
    raise ValueError('Choose an ordered date range inside the CSV coverage')
price_df = data.prices.loc[start:end, universe]
if price_df.empty:
    raise ValueError('Date range contains no observations')

rebalance_records = []
weight_fn = make_research_weight_fn(
    data=data, records=rebalance_records, llm_client=LLM_CLIENT, db_path=USAGE_DB_PATH,
)
print('Live model calls at eligible monthly rebalances; no Black-Litterman execution.')
print('EPS/rates are synthetic; observation cutoffs are not publication-time guarantees.')
agentic_bt = bt.Backtest(
    build_agentic_strategy(universe, RESEARCH_HYPOTHESIS, weight_fn=weight_fn, name='AgenticResearch'),
    price_df,
)
equal_weight_bt = bt.Backtest(build_equal_weight_strategy(universe), price_df)
result = bt.run(agentic_bt, equal_weight_bt)
result.stats


## Inspect per-rebalance views and references

Warm-up records have no research result and no LLM call. Successful records retain processed context, evidence references, validated views, warnings, and generated weights.

In [ ]:
rebalance_summary = pd.DataFrame([
    {'as_of': r.as_of, 'status': r.status, 'assets': len(r.weights),
     'observation_date': r.research.context.observation_date if r.research else None}
    for r in rebalance_records
])
view_rows = [
    {'as_of': r.as_of, **view.model_dump(), 'weight': r.weights[view.asset]}
    for r in rebalance_records if r.research
    for view in r.research.views.views
]
display(rebalance_summary)
display(pd.DataFrame(view_rows))
research_records = [r for r in rebalance_records if r.research]
if research_records:
    latest_research = research_records[-1].research
    display(pd.DataFrame({key: item.model_dump() for key, item in latest_research.context.evidence.items()}).T)


## Equity curve

Research-strength tilts versus the equal-weight baseline (indexed to 100).

In [ ]:
prices = result.prices
fig = go.Figure()
for name, colour in [('AgenticResearch', COLOUR_AGENTIC), ('EqualWeight', COLOUR_EQUAL)]:
    fig.add_trace(go.Scatter(x=prices.index, y=prices[name], name=name, mode='lines', line=dict(color=colour, width=2)))
fig.update_layout(title='Equity Curve (indexed, base=100)', xaxis_title='Date', yaxis_title='Indexed Value', template='plotly_dark', height=420)
fig.show()


## Performance metrics

These are demonstration results on synthetic research signals, not calibrated investment forecasts.

In [ ]:
stats = result.stats
metric_rows = [
    ('Total Return', 'total_return', 100), ('Ann. Return (CAGR)', 'cagr', 100),
    ('Volatility (ann.)', 'daily_vol', 100), ('Sharpe Ratio', 'daily_sharpe', 1),
    ('Sortino Ratio', 'daily_sortino', 1), ('Max Drawdown', 'max_drawdown', 100),
    ('Calmar Ratio', 'calmar', 1),
]
labels = [row[0] for row in metric_rows]
agentic_vals = [stats.loc[key, 'AgenticResearch'] * scale for _, key, scale in metric_rows]
equal_vals = [stats.loc[key, 'EqualWeight'] * scale for _, key, scale in metric_rows]
fig = go.Figure()
fig.add_trace(go.Bar(x=labels, y=equal_vals, name='EqualWeight', marker_color=COLOUR_EQUAL))
fig.add_trace(go.Bar(x=labels, y=agentic_vals, name='AgenticResearch', marker_color=COLOUR_AGENTIC))
fig.update_layout(title='Performance Metrics', barmode='group', template='plotly_dark', height=420)
fig.show()
pd.DataFrame({'AgenticResearch': agentic_vals, 'EqualWeight': equal_vals}, index=labels).round(3)
